# Step 4: Model Training

Train a **RandomForest classifier** using pre-computed features and register in Snowflake Model Registry.

## Features Used

The model is trained on **raw features + computed features** from preprocessing:
- **Raw**: Age, BMI, Heart Rate, Blood Pressure, Lab Values, etc.
- **Computed**: SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY, VITAL_SIGNS_SEVERITY

## Snowflake Services Used

| Service | Purpose |
|---------|---------|
| **ML Jobs** | Remote training on SPCS compute pools |
| **Model Registry** | Version and store models |

## Prerequisites

- Run notebooks 01-03 first (03 creates TRAINING_FEATURES and TEST_FEATURES tables)

## Imports and Configuration

In [1]:
%cd ..
%load_ext autoreload

/Users/ccaudill/src/github/snowflake-ml-e2e


In [2]:
import os
import sys
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

from snowflake.snowpark import Session
from source.configs import get_config
from source.utils import get_session

config = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)

DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
COMPUTE_WAREHOUSE = config.snowflake.warehouse

session.use_database(DB)
session.use_schema(SCHEMA)
session.use_warehouse(COMPUTE_WAREHOUSE)

print(f"Connected as: {session.get_current_user()}")
print(f"Current role: {session.get_current_role()}")
print(f"Current warehouse: {session.get_current_warehouse()}")

2026-03-24 15:09:26,001 - INFO - AST state has not been set explicitly. Defaulting to ast_enabled = True.
2026-03-24 15:09:26,208 - INFO - Snowflake Connector for Python Version: 4.3.0, Python Version: 3.11.14, Platform: macOS-15.7.4-arm64-arm-64bit
2026-03-24 15:09:26,209 - INFO - Connecting to GLOBAL Snowflake domain


Creating Session...


2026-03-24 15:09:26,999 - INFO - Snowpark Session information: 
"version" : 1.47.0,
"python.version" : 3.11.14,
"python.connector.version" : 4.3.0,
"python.connector.session.id" : 5727214342169258,
"os.name" : Darwin



Connected as: "CCAUDILL"
Current role: "SYSADMIN"
Current warehouse: "ML_DEMO_WAREHOUSE"


In [3]:
COMPUTE_POOL = config.compute.compute_pool

pool_exists = session.sql(f"SHOW COMPUTE POOLS LIKE '{COMPUTE_POOL}'").collect()
if not pool_exists:
    session.sql(f"""
        CREATE COMPUTE POOL {COMPUTE_POOL}
        MIN_NODES = 1 MAX_NODES = 1
        INSTANCE_FAMILY = CPU_X64_S
        AUTO_SUSPEND_SECS = 300
    """).collect()
    print(f"Created compute pool: {COMPUTE_POOL}")
else:
    print(f"Compute pool exists: {COMPUTE_POOL}")

Compute pool exists: ML_DEMO_COMPUTE_POOL


# Local Training

In [4]:
%autoreload
from source.train import PatientRiskTraining
from source.utils import get_feature_config

feature_config = get_feature_config(config)

trainer = PatientRiskTraining(database=DB, schema_name=SCHEMA)

trainer.train(
    train_table=f"{DB}.{SCHEMA}.TRAINING_FEATURES",
    feature_config=feature_config,
    test_table=f"{DB}.{SCHEMA}.TEST_FEATURES",
    register_model=True,
    model_name=config.model.model_name,
    target_platforms=config.model.target_platforms,
)

/Users/ccaudill/.pyenv/versions/ml-demo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-24 15:09:35,268 - INFO - Training data shape: X=(8000, 23), y=(8000,)
2026-03-24 15:09:35,269 - INFO - Training model...
2026-03-24 15:09:35,586 - INFO - Loading test data from ML_DEMO_PIPELINE_DB.HEALTHCARE.TEST_FEATURES
2026-03-24 15:09:36,806 - INFO - Evaluating on test data...
2026-03-24 15:09:36,849 - INFO - Test accuracy: 0.9350
2026-03-24 15:09:36,850 - INFO - Test F1: 0.9349
2026-03-24 15:09:39,065 - INFO - Experiment: PATIENT_RISK_MODEL_EXPERIMENT
2026-03-24 15:09:41,163 - INFO - 
Logged run 'baseline_1774382979' to experiment PATIENT_RISK_MODEL_EXPERIMENT


🏃 View run BASELINE_1774382979 at: https://app.snowflake.com/_deeplink/#/experiments/databases/ML_DEMO_PIPELINE_DB/schemas/HEALTHCARE/experiments/PATIENT_RISK_MODEL_EXPERIMENT/runs/BASELINE_1774382979
🧪 View experiment at: https://app.snowflake.com/_deeplink/#/experiments/databases/ML_DEMO_PIPELINE_DB/schemas/HEALTHCARE/experiments/PATIENT_RISK_MODEL_EXPERIMENT


2026-03-24 15:09:43,202 - INFO - 
Model saved to ML_DEMO_PIPELINE_DB.HEALTHCARE.MODEL_ARTIFACTS//tmp/model/risk_model.pkl
2026-03-24 15:09:43,203 - INFO - Metrics saved to ML_DEMO_PIPELINE_DB.HEALTHCARE.MODEL_ARTIFACTS//tmp/model/metrics.pkl
2026-03-24 15:09:43,203 - INFO - Registering model: PATIENT_RISK_MODEL


Logging model: validating model and dependencies...:   0%|          | 0/6 [00:00<?, ?it/s]

2026-03-24 15:09:44,037 - INFO - Using non-live commit model version
/Users/ccaudill/.pyenv/versions/ml-demo/lib/python3.11/site-packages/snowflake/ml/registry/_manager/model_parameter_reconciler.py:72: UserWarning: `relax_version` is not set and therefore defaulted to True. Dependency version constraints relaxed from ==x.y.z to >=x.y, <(x+1). To use specific dependency versions for compatibility, reproducibility, etc., set `options={'relax_version': False}` when logging the model.
  reconciled_options = self._reconcile_relax_version(reconciled_options, reconciled_target_platforms)
2026-03-24 15:09:47,396 - INFO - Start packaging and uploading your model. It might take some time based on the size of the model.


Logging model: creating model manifest...:  33%|███▎      | 2/6 [00:03<00:07,  1.76s/it]  

/Users/ccaudill/.pyenv/versions/ml-demo/lib/python3.11/site-packages/sklearn/ensemble/_forest.py:993: RuntimeWarning: divide by zero encountered in log
  return np.log(proba)
/Users/ccaudill/.pyenv/versions/ml-demo/lib/python3.11/site-packages/snowflake/ml/model/model_signature.py:74: UserWarning: The sample input has 10 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(
2026-03-24 15:09:48,277 - INFO - Model signatures are auto inferred as:

{'predict': ModelSignature(
    inputs=[
        FeatureSpec(dtype=DataType.INT8, name='AGE', nullable=True),
        FeatureSpec(dtype=DataType.DOUBLE, name='BMI', nullable=True),
        FeatureSpec(dtype=DataType.INT8, name='HEART_RATE', nullable=True),
        FeatureSpec(dtype=DataType.INT16, name='SYSTOLIC_BP', nullable=True),
        FeatureSpec(dtype=DataTy

Model logged successfully.: 100%|██████████| 6/6 [01:25<00:00, 14.28s/it]                          

2026-03-24 15:11:09,592 - INFO - Model registered: PATIENT_RISK_MODEL/v_20260324_150943


# ML Jobs: Remote Model Training

In [ ]:
from snowflake.ml.jobs import submit_directory

job = submit_directory(
    dir_path="source",
    entrypoint="train.py",
    compute_pool=COMPUTE_POOL,
    stage_name=f"{DB}.{SCHEMA}.JOB_PAYLOADS",
    env_vars={},
    pip_requirements=[]
)
print(f"Job submitted: {job.id}")
print(f"Status: {job.status}")

In [ ]:
print("Waiting for job to complete...")
job.wait()

print(f"\nFinal status: {job.status}")
if job.status == "DONE":
    print("\n=== Job Logs ===")
    logs = job.get_logs()
    print(logs[-3000:] if len(logs) > 3000 else logs)
else:
    print(f"Job failed: {job.status}")
    print(job.get_logs())